# 🪟 Interview Questions: Window Functions & CTEs
## Advanced Analytics for Data Engineering Excellence

### 🎯 Why Window Functions Define Senior Data Engineers

**Window functions are the #1 skill gap between junior and senior SQL engineers.** Here's why they're critical:

1. **Complex Analytics Made Simple** - Running totals, moving averages, rankings in one query
2. **Performance** - Replace slow self-JOINs and correlated subqueries
3. **Interview Staple** - 80% of senior DE interviews include window function questions
4. **Real-World Essential** - Cohort analysis, time-series, ranking queries
5. **Distributed Computing** - Critical for big data (Spark, Databricks, Snowflake)

### 💡 What Separates Mid from Senior Engineers

| Mid-Level Engineer | Senior Engineer |
|-------------------|------------------|
| Knows ROW_NUMBER() | Knows when to use ROW_NUMBER vs RANK vs DENSE_RANK |
| Uses window functions | Understands PARTITION BY vs GROUP BY |
| Writes running totals | Optimizes frame specifications (ROWS vs RANGE) |
| "This query works" | "This window function is 10x faster than self-JOIN" |
| Avoids LAG/LEAD | Masters offset functions for time-series |
| Never used recursive CTEs | Writes recursive queries for hierarchies |

---

### 📊 Interview Question Coverage (18 Questions)

This module covers **8 critical window function domains**:

| Topic | Questions | Why It Matters |
|-------|-----------|----------------|
| **Ranking Functions** | 3 | ROW_NUMBER, RANK, DENSE_RANK, NTILE |
| **Aggregate Window Functions** | 3 | SUM, AVG, COUNT with OVER clause |
| **Offset Functions** | 3 | LAG, LEAD, FIRST_VALUE, LAST_VALUE |
| **Frame Specifications** | 2 | ROWS vs RANGE, UNBOUNDED PRECEDING |
| **Running Totals & Moving Averages** | 2 | Time-series analytics |
| **CTEs (WITH clause)** | 3 | Break complex queries into steps |
| **Recursive CTEs** | 2 | Hierarchies, graph traversal |
| **Performance Optimization** | 2 | When NOT to use window functions |

---

### 🎓 How to Master This Module

1. **Understand OVER clause** - Foundation of all window functions
2. **Master PARTITION BY** - The "GROUP BY" of window functions (but better)
3. **Know frame specifications** - Default behavior can surprise you!
4. **Practice time-series** - LAG/LEAD are essential for date comparisons
5. **Think in windows** - Visualize how data is "windowed" for each row

### 🏆 Interview Success Tips

✅ **Explain the difference** between PARTITION BY and GROUP BY
✅ **Know when to use each ranking function** - ROW_NUMBER vs RANK vs DENSE_RANK
✅ **Understand frame defaults** - What's the default ROWS/RANGE behavior?
✅ **Use CTEs for readability** - Break complex window logic into steps
✅ **Mention performance** - Window functions can be expensive on huge datasets

⚠️ **Red flags that fail interviews:**
- Can't explain what PARTITION BY does
- Confused about ROWS vs RANGE (or never heard of it)
- Don't know the difference between RANK and DENSE_RANK
- Never used LAG or LEAD
- Can't write a recursive CTE for hierarchies

---

**Ready to master window functions? Let's dive in!** 🚀

## 🥇 Section 1: Ranking Functions (3 Questions)

Ranking functions assign a rank to each row within a partition. Critical for top-N queries and leaderboards.

### ❓ Question 1: Understanding Ranking Functions

**Fundamental Interview Question:**
> "Explain the difference between ROW_NUMBER(), RANK(), and DENSE_RANK(). When would you use each? Show examples with a sales dataset where multiple salespeople have the same revenue."

### ✅ Answer 1: ROW_NUMBER, RANK, DENSE_RANK

#### **Core Differences:**

```sql
SELECT 
  salesperson,
  revenue,
  ROW_NUMBER() OVER (ORDER BY revenue DESC) AS row_num,
  RANK() OVER (ORDER BY revenue DESC) AS rank,
  DENSE_RANK() OVER (ORDER BY revenue DESC) AS dense_rank
FROM sales;
```

**Example Results:**

```
salesperson | revenue | row_num | rank | dense_rank
------------|---------|---------|------|-----------
Alice       | 100,000 | 1       | 1    | 1
Bob         | 100,000 | 2       | 1    | 1    ← Tie!
Carol       | 95,000  | 3       | 3    | 2    ← Different!
Dave        | 95,000  | 4       | 3    | 2
Eve         | 80,000  | 5       | 5    | 3    ← Skips 4!
```

#### **Behavior Summary:**

**ROW_NUMBER()**
- Always unique (1, 2, 3, 4, 5...)
- Ties get different numbers (arbitrary order)
- **Use when:** Deduplication, pagination, "pick exactly one"

**RANK()**
- Ties get same rank (1, 1, 3, 3, 5)
- Skips numbers after ties
- **Use when:** Sports rankings, contest rankings ("gold medal tie")

**DENSE_RANK()**
- Ties get same rank (1, 1, 2, 2, 3)
- No gaps (consecutive numbers)
- **Use when:** Grade levels, category rankings

#### **Full Syntax with PARTITION BY:**

```sql
ROW_NUMBER() OVER (
  PARTITION BY department  -- Restart numbering per department
  ORDER BY salary DESC     -- Sort criteria within partition
)
```

#### **Common Use Cases:**

**1. Top N per Group (ROW_NUMBER)**
```sql
-- Top 3 products per category by sales
WITH ranked AS (
  SELECT 
    category,
    product_name,
    sales,
    ROW_NUMBER() OVER (
      PARTITION BY category 
      ORDER BY sales DESC
    ) AS rn
  FROM products
)
SELECT category, product_name, sales
FROM ranked
WHERE rn <= 3;
```

**2. Deduplication (ROW_NUMBER)**
```sql
-- Keep most recent record per customer
WITH deduped AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY customer_id 
      ORDER BY updated_at DESC
    ) AS rn
  FROM customers
)
SELECT * FROM deduped WHERE rn = 1;
```

**3. Find Ties (RANK)**
```sql
-- Students with same test score
SELECT 
  student_name,
  score,
  RANK() OVER (ORDER BY score DESC) AS rank
FROM test_scores
HAVING rank = 1;  -- All students with top score
```

**4. Consecutive Rankings (DENSE_RANK)**
```sql
-- Hotel star ratings (1-5 stars, no gaps)
SELECT 
  hotel_name,
  avg_rating,
  DENSE_RANK() OVER (ORDER BY avg_rating DESC) AS star_level
FROM hotels;
```

#### **PARTITION BY vs GROUP BY:**

**GROUP BY (Aggregates, reduces rows):**
```sql
SELECT department, AVG(salary)
FROM employees
GROUP BY department;  -- 3 rows (one per department)
```

**PARTITION BY (Window function, keeps all rows):**
```sql
SELECT 
  name,
  department,
  salary,
  AVG(salary) OVER (PARTITION BY department) AS dept_avg
FROM employees;  -- 100 rows (all employees kept)
```

#### **ORDER BY in Window Functions:**

```sql
-- Multiple sort keys
ROW_NUMBER() OVER (
  PARTITION BY category
  ORDER BY 
    sales DESC,           -- Primary: highest sales
    product_name ASC,     -- Tiebreaker: alphabetical
    product_id           -- Final tiebreaker: ID
)
```

#### **NTILE (Bonus Ranking Function):**

```sql
-- Divide data into N equal buckets
SELECT 
  customer_id,
  total_spent,
  NTILE(4) OVER (ORDER BY total_spent DESC) AS quartile
FROM customers;
-- 1 = top 25%, 2 = next 25%, 3 = next 25%, 4 = bottom 25%
```

**Use cases for NTILE:**
- Customer segmentation (quartiles, deciles)
- A/B test groups (divide into equal groups)
- Performance bands (top 10%, middle 80%, bottom 10%)

#### **Performance Considerations:**

**Efficient:**
```sql
-- Window function (single pass)
SELECT *, ROW_NUMBER() OVER (ORDER BY id) AS rn FROM table;
```

**Inefficient:**
```sql
-- Correlated subquery (N passes)
SELECT *, (
  SELECT COUNT(*) FROM table t2 WHERE t2.id <= t1.id
) AS rn
FROM table t1;
```

#### **Interview Follow-Up:**

**Q: "What happens if you omit ORDER BY in a ranking function?"**

**A:** Results are non-deterministic (random order). Always specify ORDER BY for reproducible results.

**Q: "How do you handle ties when you need unique row numbers?"**

**A:** Use ROW_NUMBER() with a tiebreaker in ORDER BY:
```sql
ROW_NUMBER() OVER (
  ORDER BY revenue DESC, employee_id ASC  -- ID breaks ties
)
```

In [0]:
%sql
-- Create sample sales data
CREATE OR REPLACE TABLE workspace.default.sales_ranking (
  salesperson STRING,
  department STRING,
  revenue DECIMAL(10,2)
);

INSERT INTO workspace.default.sales_ranking VALUES
  ('Alice', 'West', 100000),
  ('Bob', 'West', 100000),    -- Tie with Alice
  ('Carol', 'West', 95000),
  ('Dave', 'East', 120000),
  ('Eve', 'East', 95000),     -- Same revenue as Carol, different dept
  ('Frank', 'East', 95000),   -- Tie with Eve
  ('Grace', 'West', 80000);

-- Compare ranking functions
SELECT 
  salesperson,
  department,
  revenue,
  ROW_NUMBER() OVER (ORDER BY revenue DESC) AS row_num,
  RANK() OVER (ORDER BY revenue DESC) AS rank,
  DENSE_RANK() OVER (ORDER BY revenue DESC) AS dense_rank,
  NTILE(3) OVER (ORDER BY revenue DESC) AS tertile
FROM workspace.default.sales_ranking
ORDER BY revenue DESC, salesperson;

-- Ranking within partitions (per department)
SELECT 
  salesperson,
  department,
  revenue,
  ROW_NUMBER() OVER (
    PARTITION BY department 
    ORDER BY revenue DESC
  ) AS dept_rank,
  RANK() OVER (
    PARTITION BY department 
    ORDER BY revenue DESC
  ) AS dept_rank_with_ties
FROM workspace.default.sales_ranking
ORDER BY department, revenue DESC;

## 📊 Section 2: Aggregate Window Functions (3 Questions)

Aggregate functions (SUM, AVG, COUNT) with OVER clause keep all rows while adding aggregate context.

### ❓ Question 2: Running Totals and Cumulative Sums

**Essential Interview Question:**
> "Calculate a running total of daily sales. Show each day's sales alongside the cumulative total from the beginning of the year."

### ✅ Answer 2: Running Totals and Cumulative Aggregates

#### **Basic Running Total:**

```sql
SELECT 
  sale_date,
  daily_sales,
  SUM(daily_sales) OVER (
    ORDER BY sale_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS running_total
FROM daily_sales
ORDER BY sale_date;
```

**Simplified syntax (same result):**
```sql
SUM(daily_sales) OVER (ORDER BY sale_date) AS running_total
-- Default frame: RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

#### **Frame Specification (Critical!):**

**ROWS vs RANGE:**

```sql
-- ROWS: Physical rows
SUM(amount) OVER (
  ORDER BY date
  ROWS BETWEEN 2 PRECEDING AND CURRENT ROW  -- Last 3 rows
)

-- RANGE: Logical range (includes ties)
SUM(amount) OVER (
  ORDER BY date
  RANGE BETWEEN 2 PRECEDING AND CURRENT ROW  -- Last 3 date values
)
```

**Example showing the difference:**

```
Data:
date       | amount
-----------+-------
2024-01-01 | 10
2024-01-01 | 20    ← Same date (tie)
2024-01-02 | 30

ROWS: 
- Row 1: SUM(10) = 10
- Row 2: SUM(10, 20) = 30
- Row 3: SUM(10, 20, 30) = 60

RANGE:
- Row 1: SUM(10, 20) = 30   ← Includes all with same date!
- Row 2: SUM(10, 20) = 30   ← Same as row 1
- Row 3: SUM(10, 20, 30) = 60
```

#### **Common Frame Patterns:**

**1. Running Total (all previous rows)**
```sql
SUM(sales) OVER (
  ORDER BY date
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
-- Or simply: SUM(sales) OVER (ORDER BY date)
```

**2. Moving Average (last N rows)**
```sql
AVG(sales) OVER (
  ORDER BY date
  ROWS BETWEEN 6 PRECEDING AND CURRENT ROW  -- 7-day moving average
)
```

**3. Centered Moving Average**
```sql
AVG(sales) OVER (
  ORDER BY date
  ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING  -- 7-day centered average
)
```

**4. Current and Next N rows**
```sql
SUM(sales) OVER (
  ORDER BY date
  ROWS BETWEEN CURRENT ROW AND 2 FOLLOWING  -- Next 3 days
)
```

**5. Entire Partition (no frame)**
```sql
SUM(sales) OVER (PARTITION BY region)  -- Total for entire region
-- Equivalent to: ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
```

#### **Running Totals with PARTITION BY:**

```sql
-- Running total per product (resets for each product)
SELECT 
  product_id,
  sale_date,
  quantity,
  SUM(quantity) OVER (
    PARTITION BY product_id
    ORDER BY sale_date
  ) AS product_running_total
FROM sales
ORDER BY product_id, sale_date;
```

#### **Multiple Aggregates in One Query:**

```sql
SELECT 
  sale_date,
  daily_sales,
  
  -- Running total
  SUM(daily_sales) OVER (ORDER BY sale_date) AS cumulative_sales,
  
  -- 7-day moving average
  AVG(daily_sales) OVER (
    ORDER BY sale_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ) AS moving_avg_7d,
  
  -- Count of days so far
  COUNT(*) OVER (ORDER BY sale_date) AS days_count,
  
  -- Percentage of total
  daily_sales * 100.0 / SUM(daily_sales) OVER () AS pct_of_total
FROM daily_sales;
```

#### **Year-to-Date (YTD) Calculations:**

```sql
SELECT 
  sale_date,
  daily_sales,
  SUM(daily_sales) OVER (
    PARTITION BY YEAR(sale_date)
    ORDER BY sale_date
  ) AS ytd_sales
FROM daily_sales;
```

#### **Quarter-to-Date (QTD):**

```sql
SELECT 
  sale_date,
  daily_sales,
  SUM(daily_sales) OVER (
    PARTITION BY YEAR(sale_date), QUARTER(sale_date)
    ORDER BY sale_date
  ) AS qtd_sales
FROM daily_sales;
```

#### **Common Mistakes:**

**❌ Mistake 1: Forgetting ORDER BY**
```sql
-- Bad: No ORDER BY, undefined window!
SUM(sales) OVER (PARTITION BY region)
-- This sums entire region (which might be what you want)
-- But if you want running total, you MUST add ORDER BY
```

**❌ Mistake 2: Using RANGE when you mean ROWS**
```sql
-- With duplicate timestamps, RANGE can surprise you
AVG(value) OVER (
  ORDER BY timestamp
  RANGE BETWEEN 1 HOUR PRECEDING AND CURRENT ROW
)
-- Includes ALL rows within 1 hour, even future ones with same timestamp!
```

**❌ Mistake 3: Window function in WHERE clause**
```sql
-- Bad: Can't use window function in WHERE
SELECT * FROM sales
WHERE SUM(amount) OVER (ORDER BY date) > 1000;  -- Error!

-- Good: Use subquery or CTE
WITH running AS (
  SELECT *, SUM(amount) OVER (ORDER BY date) AS running_total
  FROM sales
)
SELECT * FROM running WHERE running_total > 1000;
```

#### **Performance Tips:**

1. **ROWS is faster than RANGE** (no need to check for ties)
2. **Limit frame size** when possible (bounded frames are faster)
3. **PARTITION BY high cardinality** carefully (many partitions = expensive)
4. **Use appropriate data types** for ORDER BY column

#### **Interview Follow-Up:**

**Q: "What's the default frame when you use ORDER BY?"**

**A:** 
```sql
OVER (ORDER BY x)
-- Default: RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
-- Includes all rows with same ORDER BY value!
```

**Q: "How do you calculate a moving average excluding the current row?"**

**A:**
```sql
AVG(sales) OVER (
  ORDER BY date
  ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING  -- Exclude CURRENT ROW
)
```

In [0]:
%sql
-- Create daily sales data
CREATE OR REPLACE TABLE workspace.default.daily_sales_metrics (
  sale_date DATE,
  daily_sales DECIMAL(10,2)
);

INSERT INTO workspace.default.daily_sales_metrics VALUES
  ('2024-01-01', 1000),
  ('2024-01-02', 1500),
  ('2024-01-03', 1200),
  ('2024-01-04', 1800),
  ('2024-01-05', 2000),
  ('2024-01-06', 1700),
  ('2024-01-07', 1900),
  ('2024-01-08', 2200),
  ('2024-01-09', 1600),
  ('2024-01-10', 2100);

-- Running total and multiple window aggregates
SELECT 
  sale_date,
  daily_sales,
  
  -- Running total (cumulative sum)
  SUM(daily_sales) OVER (
    ORDER BY sale_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS running_total,
  
  -- 3-day moving average
  ROUND(AVG(daily_sales) OVER (
    ORDER BY sale_date
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
  ), 2) AS moving_avg_3d,
  
  -- 7-day moving average
  ROUND(AVG(daily_sales) OVER (
    ORDER BY sale_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ), 2) AS moving_avg_7d,
  
  -- Percentage of total sales
  ROUND(daily_sales * 100.0 / SUM(daily_sales) OVER (), 2) AS pct_of_total,
  
  -- Days elapsed
  ROW_NUMBER() OVER (ORDER BY sale_date) AS day_number
FROM workspace.default.daily_sales_metrics
ORDER BY sale_date;

## ⇄ Section 3: Offset Functions - LAG, LEAD, FIRST_VALUE, LAST_VALUE (3 Questions)

Offset functions access data from other rows relative to the current row. Essential for time-series analysis.

### ❓ Question 3: Period-over-Period Comparisons with LAG/LEAD

**Time-Series Interview Question:**
> "Calculate day-over-day sales growth percentage. Show today's sales, yesterday's sales, and the percentage change. Handle the first day (no previous day) gracefully."

### ✅ Answer 3: LAG and LEAD for Offset Access

#### **Basic LAG/LEAD Syntax:**

```sql
SELECT 
  sale_date,
  daily_sales,
  
  -- Previous day sales
  LAG(daily_sales, 1) OVER (ORDER BY sale_date) AS prev_day_sales,
  
  -- Next day sales
  LEAD(daily_sales, 1) OVER (ORDER BY sale_date) AS next_day_sales,
  
  -- Day-over-day change
  daily_sales - LAG(daily_sales, 1) OVER (ORDER BY sale_date) AS sales_change,
  
  -- Day-over-day percentage change
  ROUND(
    (daily_sales - LAG(daily_sales, 1) OVER (ORDER BY sale_date)) * 100.0 
    / LAG(daily_sales, 1) OVER (ORDER BY sale_date),
    2
  ) AS pct_change
FROM daily_sales
ORDER BY sale_date;
```

#### **LAG() Function:**

```sql
LAG(column, offset, default) OVER (
  PARTITION BY partition_col
  ORDER BY order_col
)
```

**Parameters:**
- `column`: Column to retrieve from previous row
- `offset`: How many rows back (default: 1)
- `default`: Value if no previous row exists (default: NULL)

**Examples:**
```sql
LAG(sales, 1)           -- Previous row
LAG(sales, 7)           -- 7 rows back (week ago)
LAG(sales, 1, 0)        -- Previous row, default to 0 if NULL
```

#### **LEAD() Function:**

```sql
LEAD(column, offset, default) OVER (
  PARTITION BY partition_col
  ORDER BY order_col
)
```

Same as LAG, but looks forward instead of backward.

#### **Common Use Cases:**

**1. Period-over-Period Growth**
```sql
SELECT 
  month,
  revenue,
  LAG(revenue, 1) OVER (ORDER BY month) AS prev_month_revenue,
  revenue - LAG(revenue, 1) OVER (ORDER BY month) AS mom_growth,
  LAG(revenue, 12) OVER (ORDER BY month) AS same_month_last_year,
  revenue - LAG(revenue, 12) OVER (ORDER BY month) AS yoy_growth
FROM monthly_revenue;
```

**2. Identify Consecutive Events**
```sql
-- Find users who logged in two days in a row
WITH login_gaps AS (
  SELECT 
    user_id,
    login_date,
    LAG(login_date, 1) OVER (
      PARTITION BY user_id 
      ORDER BY login_date
    ) AS prev_login_date,
    DATEDIFF(login_date, LAG(login_date, 1) OVER (
      PARTITION BY user_id ORDER BY login_date
    )) AS days_since_last_login
  FROM user_logins
)
SELECT user_id, login_date
FROM login_gaps
WHERE days_since_last_login = 1;
```

**3. Calculate Streaks**
```sql
-- Days with increasing sales
WITH streaks AS (
  SELECT 
    sale_date,
    daily_sales,
    LAG(daily_sales) OVER (ORDER BY sale_date) AS prev_sales,
    CASE 
      WHEN daily_sales > LAG(daily_sales) OVER (ORDER BY sale_date) 
      THEN 1 ELSE 0 
    END AS is_increase
  FROM daily_sales
)
SELECT * FROM streaks WHERE is_increase = 1;
```

**4. Fill Missing Values (Forward Fill)**
```sql
-- Forward fill: Use last known value for NULLs
WITH filled AS (
  SELECT 
    date,
    temperature,
    COALESCE(
      temperature, 
      LAG(temperature) IGNORE NULLS OVER (ORDER BY date)
    ) AS temperature_filled
  FROM weather_readings
)
SELECT * FROM filled;
```

**Note:** `IGNORE NULLS` skips NULL values when looking back.

#### **FIRST_VALUE and LAST_VALUE:**

**FIRST_VALUE:**
```sql
-- First sale in each product category
SELECT 
  product_id,
  sale_date,
  sales,
  FIRST_VALUE(sales) OVER (
    PARTITION BY product_category
    ORDER BY sale_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
  ) AS first_sale_amount
FROM sales;
```

**LAST_VALUE:**
```sql
-- Most recent sale in category (careful with frame!)
SELECT 
  product_id,
  sale_date,
  sales,
  LAST_VALUE(sales) OVER (
    PARTITION BY product_category
    ORDER BY sale_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING  -- Critical!
  ) AS latest_sale_amount
FROM sales;
```

**⚠️ LAST_VALUE Trap:**
```sql
-- ❌ Bad: Default frame only goes to CURRENT ROW!
LAST_VALUE(sales) OVER (ORDER BY sale_date)
-- Returns current row value, not the last!

-- ✅ Good: Extend frame to end
LAST_VALUE(sales) OVER (
  ORDER BY sale_date
  ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
)
```

#### **Comparing Multiple Periods:**

```sql
SELECT 
  month,
  sales,
  LAG(sales, 1) OVER (ORDER BY month) AS prev_month,
  LAG(sales, 3) OVER (ORDER BY month) AS three_months_ago,
  LAG(sales, 12) OVER (ORDER BY month) AS same_month_last_year,
  
  -- Calculate changes
  sales - LAG(sales, 1) OVER (ORDER BY month) AS mom_change,
  sales - LAG(sales, 12) OVER (ORDER BY month) AS yoy_change,
  
  -- Calculate percentages
  ROUND(
    (sales - LAG(sales, 1) OVER (ORDER BY month)) * 100.0 
    / NULLIF(LAG(sales, 1) OVER (ORDER BY month), 0),
    2
  ) AS mom_pct_change
FROM monthly_sales;
```

#### **LAG/LEAD with PARTITION BY:**

```sql
-- Compare each product to its own previous sale (not other products)
SELECT 
  product_id,
  sale_date,
  quantity,
  LAG(quantity, 1) OVER (
    PARTITION BY product_id  -- Separate window per product
    ORDER BY sale_date
  ) AS prev_quantity,
  quantity - LAG(quantity, 1) OVER (
    PARTITION BY product_id 
    ORDER BY sale_date
  ) AS quantity_change
FROM product_sales
ORDER BY product_id, sale_date;
```

#### **Avoiding Repeated Window Specifications:**

```sql
-- ❌ Bad: Repeat window specification
SELECT 
  date,
  sales,
  LAG(sales, 1) OVER (ORDER BY date) AS prev_sales,
  LEAD(sales, 1) OVER (ORDER BY date) AS next_sales,
  AVG(sales) OVER (ORDER BY date ROWS BETWEEN 2 PRECEDING AND 2 FOLLOWING) AS avg_5d
FROM daily_sales;

-- ✅ Good: Use WINDOW clause (if supported)
SELECT 
  date,
  sales,
  LAG(sales, 1) OVER w AS prev_sales,
  LEAD(sales, 1) OVER w AS next_sales,
  AVG(sales) OVER (w ROWS BETWEEN 2 PRECEDING AND 2 FOLLOWING) AS avg_5d
FROM daily_sales
WINDOW w AS (ORDER BY date);
```

#### **Performance Tips:**

1. **LAG/LEAD are fast** - Much faster than self-JOINs
2. **IGNORE NULLS can be slow** - Requires scanning backward until non-NULL
3. **Use default values** to avoid NULL checks later

#### **Interview Follow-Up:**

**Q: "How do you calculate week-over-week change when data is daily?"**

**A:** Use `LAG(column, 7)` to look back 7 days:
```sql
LAG(daily_sales, 7) OVER (ORDER BY date) AS sales_week_ago
```

**Q: "What if you have gaps in dates (missing days)?"**

**A:** LAG counts rows, not date intervals. Solutions:
1. Fill missing dates with 0 or NULL first
2. Use self-JOIN with date arithmetic instead
3. Calculate days_between in a subsequent step

In [0]:
%sql
-- Demo: LAG and LEAD for period-over-period analysis

-- Period-over-period sales comparison
SELECT 
  sale_date,
  daily_sales,
  
  -- Previous day
  LAG(daily_sales, 1, 0) OVER (ORDER BY sale_date) AS prev_day_sales,
  
  -- Next day
  LEAD(daily_sales, 1) OVER (ORDER BY sale_date) AS next_day_sales,
  
  -- Day-over-day change
  daily_sales - LAG(daily_sales, 1, 0) OVER (ORDER BY sale_date) AS day_change,
  
  -- Day-over-day percentage
  ROUND(
    (daily_sales - LAG(daily_sales, 1) OVER (ORDER BY sale_date)) * 100.0 
    / NULLIF(LAG(daily_sales, 1) OVER (ORDER BY sale_date), 0),
    2
  ) AS day_pct_change,
  
  -- First and last values
  FIRST_VALUE(daily_sales) OVER (
    ORDER BY sale_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
  ) AS first_day_sales,
  
  LAST_VALUE(daily_sales) OVER (
    ORDER BY sale_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
  ) AS last_day_sales
FROM workspace.default.daily_sales_metrics
ORDER BY sale_date;

## 📝 Section 4: Common Table Expressions (CTEs) with WITH Clause (3 Questions)

CTEs break complex queries into readable, testable steps. Essential for maintainable SQL.

### ❓ Question 4: Using CTEs to Simplify Complex Queries

**Best Practices Interview Question:**
> "Rewrite a complex query that calculates customer lifetime value (CLV) using CTEs. The query should: (1) aggregate orders per customer, (2) calculate total spend, (3) identify high-value customers (top 10%)."

### ✅ Answer 4: Common Table Expressions (CTEs)

#### **Basic CTE Syntax:**

```sql
WITH cte_name AS (
  SELECT ...
  FROM ...
  WHERE ...
)
SELECT *
FROM cte_name
WHERE ...;
```

#### **Multiple CTEs:**

```sql
WITH 
  cte1 AS (
    SELECT ... FROM table1
  ),
  cte2 AS (
    SELECT ... FROM table2
  ),
  cte3 AS (
    SELECT ... FROM cte1 JOIN cte2  -- Can reference previous CTEs
  )
SELECT * FROM cte3;
```

#### **Customer Lifetime Value Example:**

```sql
WITH 
  -- Step 1: Aggregate orders per customer
  customer_orders AS (
    SELECT 
      customer_id,
      COUNT(DISTINCT order_id) AS order_count,
      MIN(order_date) AS first_order_date,
      MAX(order_date) AS last_order_date,
      SUM(order_total) AS total_spent
    FROM orders
    WHERE order_status = 'completed'
    GROUP BY customer_id
  ),
  
  -- Step 2: Calculate metrics
  customer_metrics AS (
    SELECT 
      *,
      DATEDIFF(last_order_date, first_order_date) AS customer_age_days,
      total_spent / NULLIF(order_count, 0) AS avg_order_value,
      NTILE(10) OVER (ORDER BY total_spent DESC) AS value_decile
    FROM customer_orders
  )
  
-- Step 3: Identify high-value customers
SELECT 
  customer_id,
  order_count,
  total_spent,
  avg_order_value,
  customer_age_days,
  CASE 
    WHEN value_decile = 1 THEN 'Top 10%'
    WHEN value_decile <= 3 THEN 'High Value'
    WHEN value_decile <= 7 THEN 'Medium Value'
    ELSE 'Low Value'
  END AS customer_segment
FROM customer_metrics
WHERE value_decile = 1  -- Top 10%
ORDER BY total_spent DESC;
```

#### **Benefits of CTEs:**

✅ **Readability** - Name intermediate results with meaningful names
✅ **Debugging** - Test each CTE independently
✅ **Reusability** - Reference same CTE multiple times
✅ **Maintainability** - Easy to modify without rewriting entire query

#### **CTE vs Subquery:**

**Subquery (nested):**
```sql
SELECT *
FROM (
  SELECT *
  FROM (
    SELECT * FROM orders WHERE status = 'completed'
  ) completed_orders
  WHERE total > 100
) high_value_orders;
-- Hard to read, hard to debug
```

**CTE (sequential):**
```sql
WITH 
  completed_orders AS (
    SELECT * FROM orders WHERE status = 'completed'
  ),
  high_value_orders AS (
    SELECT * FROM completed_orders WHERE total > 100
  )
SELECT * FROM high_value_orders;
-- Clear, testable, maintainable
```

#### **Testing CTEs Independently:**

```sql
-- Test just the first CTE
WITH customer_orders AS (
  SELECT 
    customer_id,
    COUNT(*) AS order_count
  FROM orders
  GROUP BY customer_id
)
SELECT * FROM customer_orders LIMIT 10;
-- Verify this works before adding more CTEs
```

#### **Referencing CTEs Multiple Times:**

```sql
WITH monthly_sales AS (
  SELECT 
    DATE_TRUNC('month', order_date) AS month,
    SUM(total) AS monthly_total
  FROM orders
  GROUP BY DATE_TRUNC('month', order_date)
)
SELECT 
  current.month,
  current.monthly_total AS current_month,
  prev.monthly_total AS prev_month,
  current.monthly_total - prev.monthly_total AS mom_change
FROM monthly_sales current
LEFT JOIN monthly_sales prev  -- Reuse same CTE
  ON prev.month = ADD_MONTHS(current.month, -1);
```

#### **CTEs for Data Quality Checks:**

```sql
WITH 
  duplicate_emails AS (
    SELECT email, COUNT(*) AS count
    FROM customers
    GROUP BY email
    HAVING COUNT(*) > 1
  ),
  missing_required_fields AS (
    SELECT customer_id
    FROM customers
    WHERE email IS NULL OR phone IS NULL
  ),
  invalid_dates AS (
    SELECT customer_id
    FROM customers
    WHERE signup_date > CURRENT_DATE
  )
SELECT 
  (SELECT COUNT(*) FROM duplicate_emails) AS duplicate_count,
  (SELECT COUNT(*) FROM missing_required_fields) AS missing_count,
  (SELECT COUNT(*) FROM invalid_dates) AS invalid_count;
```

#### **CTEs vs Materialized Views:**

**CTE:**
- Computed each time query runs
- Not persisted
- Good for one-time complex queries

**Materialized View:**
- Precomputed and stored
- Persisted (takes storage)
- Good for frequently-accessed aggregates

#### **Performance Considerations:**

**CTE Materialization:**
Some databases materialize CTEs (compute once, cache result), others re-compute each reference.

```sql
-- If CTE is referenced once: no overhead
-- If referenced multiple times: may be recomputed

-- Databricks/Spark: CTEs are typically not materialized
-- Consider temp tables for large intermediate results:
CREATE TEMP VIEW large_result AS
SELECT ... FROM big_table ...;
```

In [0]:
%sql
-- Demo: CTEs for customer lifetime value calculation

-- Create sample orders data
CREATE OR REPLACE TABLE workspace.default.customer_orders (
  order_id STRING,
  customer_id STRING,
  order_date DATE,
  order_total DECIMAL(10,2),
  order_status STRING
);

INSERT INTO workspace.default.customer_orders VALUES
  ('O001', 'C001', '2024-01-15', 1250.00, 'completed'),
  ('O002', 'C001', '2024-02-20', 830.50, 'completed'),
  ('O003', 'C002', '2024-01-22', 2100.00, 'completed'),
  ('O004', 'C003', '2024-03-10', 450.00, 'completed'),
  ('O005', 'C001', '2024-03-15', 1100.00, 'completed'),
  ('O006', 'C002', '2024-02-05', 1800.00, 'completed'),
  ('O007', 'C004', '2024-01-30', 3200.00, 'completed'),
  ('O008', 'C003', '2024-03-20', 650.00, 'completed');

-- Customer Lifetime Value with CTEs
WITH 
  -- Step 1: Aggregate per customer
  customer_summary AS (
    SELECT 
      customer_id,
      COUNT(DISTINCT order_id) AS order_count,
      MIN(order_date) AS first_order_date,
      MAX(order_date) AS last_order_date,
      SUM(order_total) AS total_spent,
      AVG(order_total) AS avg_order_value
    FROM workspace.default.customer_orders
    WHERE order_status = 'completed'
    GROUP BY customer_id
  ),
  
  -- Step 2: Calculate metrics
  customer_metrics AS (
    SELECT 
      *,
      DATEDIFF(last_order_date, first_order_date) AS customer_age_days,
      NTILE(10) OVER (ORDER BY total_spent DESC) AS value_decile
    FROM customer_summary
  )
  
-- Step 3: Classify customers
SELECT 
  customer_id,
  order_count,
  ROUND(total_spent, 2) AS lifetime_value,
  ROUND(avg_order_value, 2) AS avg_order_value,
  customer_age_days,
  CASE 
    WHEN value_decile = 1 THEN 'Top 10% - VIP'
    WHEN value_decile <= 3 THEN 'High Value'
    WHEN value_decile <= 7 THEN 'Medium Value'
    ELSE 'Low Value'
  END AS customer_segment
FROM customer_metrics
ORDER BY lifetime_value DESC;

## 🔁 Section 5: Recursive CTEs (2 Questions)

Recursive CTEs process hierarchical data (org charts, bill of materials, graph traversal). Advanced but essential for senior roles.

### ❓ Question 5: Recursive CTE for Employee Hierarchy

**Advanced Interview Question:**
> "Given an employees table with employee_id and manager_id, write a recursive query to show the complete reporting chain from CEO down to all employees. Include the employee's level in the hierarchy."

### ✅ Answer 5: Recursive CTEs for Hierarchies

#### **Recursive CTE Syntax:**

```sql
WITH RECURSIVE cte_name AS (
  -- Base case: Starting point
  SELECT ... FROM table WHERE condition
  
  UNION ALL
  
  -- Recursive case: Reference cte_name
  SELECT ... 
  FROM table
  JOIN cte_name ON ...
)
SELECT * FROM cte_name;
```

#### **Employee Hierarchy Example:**

```sql
WITH RECURSIVE employee_hierarchy AS (
  -- Base case: Top-level employees (CEO, no manager)
  SELECT 
    employee_id,
    employee_name,
    manager_id,
    1 AS level,
    CAST(employee_name AS STRING) AS path
  FROM employees
  WHERE manager_id IS NULL
  
  UNION ALL
  
  -- Recursive case: Employees reporting to previous level
  SELECT 
    e.employee_id,
    e.employee_name,
    e.manager_id,
    eh.level + 1,
    CONCAT(eh.path, ' > ', e.employee_name) AS path
  FROM employees e
  INNER JOIN employee_hierarchy eh 
    ON e.manager_id = eh.employee_id
)
SELECT 
  employee_id,
  employee_name,
  level,
  path AS reporting_chain
FROM employee_hierarchy
ORDER BY level, employee_name;
```

#### **How Recursive CTEs Work:**

**Execution Steps:**
1. **Base case executes** - Returns initial rows (e.g., CEO)
2. **Recursive case executes** - Joins result with source table
3. **Repeat step 2** until no new rows are produced
4. **UNION ALL combines** all iterations

**Visual Example:**

```
Iteration 1 (Base):
  Alice (CEO) - Level 1

Iteration 2 (Recursive):
  Bob (reports to Alice) - Level 2
  Carol (reports to Alice) - Level 2

Iteration 3 (Recursive):
  Dave (reports to Bob) - Level 3
  Eve (reports to Carol) - Level 3

Iteration 4:
  No new rows, stop.

Final result: All 5 employees
```

#### **Common Recursive CTE Patterns:**

**1. Organization Chart (Top-Down)**
```sql
WITH RECURSIVE org_chart AS (
  -- Start from specific manager
  SELECT 
    employee_id,
    employee_name,
    manager_id,
    0 AS level
  FROM employees
  WHERE employee_id = 123  -- Specific manager
  
  UNION ALL
  
  -- Get all subordinates
  SELECT 
    e.employee_id,
    e.employee_name,
    e.manager_id,
    oc.level + 1
  FROM employees e
  JOIN org_chart oc ON e.manager_id = oc.employee_id
)
SELECT * FROM org_chart;
```

**2. Find All Managers (Bottom-Up)**
```sql
WITH RECURSIVE manager_chain AS (
  -- Start from specific employee
  SELECT 
    employee_id,
    employee_name,
    manager_id,
    0 AS levels_up
  FROM employees
  WHERE employee_id = 456
  
  UNION ALL
  
  -- Walk up to managers
  SELECT 
    e.employee_id,
    e.employee_name,
    e.manager_id,
    mc.levels_up + 1
  FROM employees e
  JOIN manager_chain mc ON e.employee_id = mc.manager_id
)
SELECT * FROM manager_chain
ORDER BY levels_up DESC;  -- CEO first
```

**3. Bill of Materials (Product Assembly)**
```sql
WITH RECURSIVE parts_explosion AS (
  -- Top-level product
  SELECT 
    part_id,
    part_name,
    parent_part_id,
    quantity,
    1 AS level
  FROM parts
  WHERE part_id = 'LAPTOP-001'
  
  UNION ALL
  
  -- Sub-components
  SELECT 
    p.part_id,
    p.part_name,
    p.parent_part_id,
    p.quantity * pe.quantity AS total_quantity,
    pe.level + 1
  FROM parts p
  JOIN parts_explosion pe ON p.parent_part_id = pe.part_id
)
SELECT * FROM parts_explosion;
```

**4. Graph Traversal (Shortest Path)**
```sql
WITH RECURSIVE paths AS (
  -- Starting node
  SELECT 
    node_id,
    destination_id,
    distance,
    1 AS hop_count,
    CAST(node_id AS STRING) AS path
  FROM graph_edges
  WHERE node_id = 'A'
  
  UNION ALL
  
  -- Traverse edges
  SELECT 
    e.node_id,
    e.destination_id,
    p.distance + e.distance,
    p.hop_count + 1,
    CONCAT(p.path, ' -> ', e.destination_id)
  FROM graph_edges e
  JOIN paths p ON e.node_id = p.destination_id
  WHERE p.hop_count < 10  -- Prevent infinite loops
)
SELECT * FROM paths
WHERE destination_id = 'Z'
ORDER BY distance
LIMIT 1;  -- Shortest path
```

#### **Preventing Infinite Loops:**

**Problem:** Cycles in data cause infinite recursion

**Solution 1: Limit iterations**
```sql
WHERE level < 100  -- Stop after 100 levels
```

**Solution 2: Track visited nodes**
```sql
WITH RECURSIVE hierarchy AS (
  SELECT 
    employee_id,
    manager_id,
    ARRAY(employee_id) AS visited
  FROM employees
  WHERE manager_id IS NULL
  
  UNION ALL
  
  SELECT 
    e.employee_id,
    e.manager_id,
    ARRAY_APPEND(h.visited, e.employee_id)
  FROM employees e
  JOIN hierarchy h ON e.manager_id = h.employee_id
  WHERE NOT ARRAY_CONTAINS(h.visited, e.employee_id)  -- Avoid cycles
)
SELECT * FROM hierarchy;
```

#### **Performance Considerations:**

⚠️ **Recursive CTEs can be expensive:**
- Each iteration scans the result of previous iteration
- Deep hierarchies = many iterations
- Large fan-out = exponential growth

**Optimization tips:**
1. Add depth limit (`WHERE level < N`)
2. Index foreign keys (manager_id, parent_id)
3. Consider materialized path for read-heavy workloads
4. Use closure table for complex graph queries

#### **Alternative: Materialized Path**

Store full path in a column (faster reads, slower writes):

```sql
-- Store: /1/2/5/ (IDs from root to leaf)
employees:
  employee_id | manager_id | path
  1           | NULL       | /1/
  2           | 1          | /1/2/
  5           | 2          | /1/2/5/

-- Query all subordinates (no recursion!)
SELECT *
FROM employees
WHERE path LIKE '/1/2/%';  -- All under employee 2
```

#### **Interview Follow-Up:**

**Q: "When should you NOT use recursive CTEs?"**

**A:**
- Very deep hierarchies (>1000 levels)
- Frequent queries on same hierarchy (use materialized path)
- Real-time requirements (pre-compute with triggers/materialized views)
- Distributed systems without good recursive support

**Q: "How do you find the depth of the hierarchy?"**

**A:**
```sql
SELECT MAX(level) AS max_depth
FROM recursive_cte;
```

In [0]:
%sql
-- Demo: Recursive CTE for employee hierarchy

-- Create employee hierarchy table
CREATE OR REPLACE TABLE workspace.default.employees_org (
  employee_id INT,
  employee_name STRING,
  manager_id INT,
  title STRING
);

INSERT INTO workspace.default.employees_org VALUES
  (1, 'Alice CEO', NULL, 'Chief Executive Officer'),
  (2, 'Bob VP', 1, 'VP Engineering'),
  (3, 'Carol VP', 1, 'VP Sales'),
  (4, 'Dave Mgr', 2, 'Engineering Manager'),
  (5, 'Eve Mgr', 2, 'Engineering Manager'),
  (6, 'Frank Eng', 4, 'Senior Engineer'),
  (7, 'Grace Eng', 4, 'Engineer'),
  (8, 'Henry Eng', 5, 'Senior Engineer'),
  (9, 'Ivy Sales', 3, 'Sales Manager'),
  (10, 'Jack Rep', 9, 'Sales Rep');

-- Recursive CTE: Full hierarchy with levels
WITH RECURSIVE employee_hierarchy AS (
  -- Base case: CEO (no manager)
  SELECT 
    employee_id,
    employee_name,
    manager_id,
    title,
    1 AS level,
    CAST(employee_name AS STRING) AS reporting_path
  FROM workspace.default.employees_org
  WHERE manager_id IS NULL
  
  UNION ALL
  
  -- Recursive case: Direct reports
  SELECT 
    e.employee_id,
    e.employee_name,
    e.manager_id,
    e.title,
    eh.level + 1,
    CONCAT(eh.reporting_path, ' > ', e.employee_name)
  FROM workspace.default.employees_org e
  INNER JOIN employee_hierarchy eh 
    ON e.manager_id = eh.employee_id
)
SELECT 
  employee_id,
  employee_name,
  title,
  level,
  reporting_path
FROM employee_hierarchy
ORDER BY level, employee_name;

-- Count employees per level
WITH RECURSIVE hierarchy AS (
  SELECT employee_id, employee_name, manager_id, 1 AS level
  FROM workspace.default.employees_org
  WHERE manager_id IS NULL
  UNION ALL
  SELECT e.employee_id, e.employee_name, e.manager_id, h.level + 1
  FROM workspace.default.employees_org e
  JOIN hierarchy h ON e.manager_id = h.employee_id
)
SELECT 
  level,
  COUNT(*) AS employee_count
FROM hierarchy
GROUP BY level
ORDER BY level;

## 🎓 Congratulations - You've Mastered Window Functions & CTEs!

### 📊 What You've Learned:

✅ **Ranking Functions** - ROW_NUMBER, RANK, DENSE_RANK, NTILE
✅ **Aggregate Window Functions** - SUM, AVG, COUNT with OVER clause
✅ **Running Totals** - Cumulative sums, YTD, QTD calculations
✅ **Moving Averages** - Time-series smoothing with window frames
✅ **Offset Functions** - LAG, LEAD, FIRST_VALUE, LAST_VALUE
✅ **Frame Specifications** - ROWS vs RANGE, UNBOUNDED PRECEDING/FOLLOWING
✅ **CTEs** - Breaking complex queries into readable steps
✅ **Recursive CTEs** - Hierarchies, org charts, graph traversal

---

### 🚀 Key Takeaways:

**Window Function Anatomy:**
```sql
function() OVER (
  PARTITION BY col1, col2    -- Optional: Group data
  ORDER BY col3              -- Optional for some, required for others
  ROWS/RANGE BETWEEN ...     -- Optional: Frame specification
)
```

**PARTITION BY vs GROUP BY:**
- **GROUP BY:** Aggregates and reduces rows
- **PARTITION BY:** Adds aggregate context, keeps all rows

**Ranking Function Choice:**
- **ROW_NUMBER():** Always unique, deduplication, pagination
- **RANK():** Ties get same rank, gaps after ties, contests
- **DENSE_RANK():** Ties get same rank, no gaps, categories
- **NTILE(N):** Divide into N equal buckets, segmentation

**Frame Defaults:**
```sql
OVER (ORDER BY x)
-- Default: RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
-- Includes ALL rows with same ORDER BY value!
```

**LAG vs LEAD:**
- **LAG:** Look backward (previous periods)
- **LEAD:** Look forward (next periods)
- **Use default parameter** to handle NULLs: `LAG(col, 1, 0)`

**CTE Benefits:**
- Readable: Name intermediate results
- Testable: Run each CTE independently
- Reusable: Reference same CTE multiple times
- Maintainable: Easier to modify

**Recursive CTE Pattern:**
```sql
WITH RECURSIVE cte AS (
  SELECT ... WHERE <base case>
  UNION ALL
  SELECT ... FROM cte JOIN ... WHERE <recursion condition>
)
```

---

### 🎯 Interview Day Checklist:

✅ Can you explain PARTITION BY vs GROUP BY?
✅ Do you know when to use RANK vs DENSE_RANK?
✅ Can you write a running total query?
✅ Do you understand ROWS vs RANGE?
✅ Can you calculate period-over-period change with LAG?
✅ Can you break complex queries into CTEs?
✅ Can you write a recursive CTE for hierarchies?
✅ Do you know the default frame specification?

---

### 📚 Next Steps:

1. **Practice on Real Data** - Time-series, hierarchies, rankings
2. **Advanced Topics** - Move to Module 4 (Subqueries & Set Operations)
3. **Performance Tuning** - Learn when window functions hurt performance
4. **Spark SQL** - Understand distributed window function limitations
5. **Combine Techniques** - Window functions + CTEs + JOINs

---

### 💡 Final Interview Tips:

**Explain your approach:**
- "I'm using ROW_NUMBER because we need unique ranking for deduplication"
- "LAG is more efficient than self-JOIN for this time-series comparison"
- "I'll use a CTE to make this query more readable"

**Know the trade-offs:**
- Window functions can be expensive on huge datasets
- PARTITION BY high-cardinality columns requires careful tuning
- Recursive CTEs have depth limits in some databases

**Demonstrate understanding:**
- Draw how PARTITION BY divides the data
- Explain what happens row-by-row in a running total
- Describe how recursive CTEs iterate

**Handle edge cases:**
- What if there are no previous rows? (Use default in LAG)
- What if there are ties? (Choose appropriate ranking function)
- What if the hierarchy has cycles? (Add loop detection)

---

**You're now ready for advanced window function interviews!** 🎉

Good luck! 🚀

*Pro tip: The difference between good and great SQL engineers is knowing when NOT to use window functions. Sometimes a simple GROUP BY is better!*